# VPIN across de Prado Information Bars — GARCH-adjusted, one dashboard per bar

This notebook is a faithful **replica** of the original 1-minute VPIN dashboard, generalised so that
the **exact same graph** (same colours, same layout — *price on the left, confidence on the right*)
is produced for **every information bar defined by López de Prado** (*Advances in Financial Machine
Learning*, Ch. 2):

| Bar | Sampling threshold | User name |
|-----|--------------------|-----------|
| **Volume bars** | cumulative **share volume** | *volume info bar* |
| **Dollar bars** | cumulative **price × volume** (equity value traded) | *equity* |
| **Tick bars**   | cumulative **number of ticks** (price updates) | *price* |

Key properties of this build:

* **GARCH(1,1)/EWMA volatility adjusts the bar size** — during high-volatility (fast information
  flow) the bar threshold shrinks, so we sample more often; during quiet periods it grows.
* **Only the VPIN is computed on the information-bar clock** — the Bulk-Volume-Classification (BVC),
  the VPIN buckets and its rolling window are all measured in *bars*, exactly as de Prado defines the
  volume-synchronised PIN.
* **Moving averages are added with respect to the information bars** — fast/slow price MAs and a VPIN
  MA, all with windows counted in *bars*.
* Choose one bar or show them all: the **same dashboard is rendered one after another** for each
  selected information bar.


In [ ]:
# --- install (Colab / fresh envs); harmless if already present ---
try:
    import plotly, scipy, statsmodels, yfinance  # noqa
except Exception:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "plotly", "scipy", "statsmodels", "yfinance", "pandas", "numpy"])

import numpy as np
import pandas as pd
import scipy.stats as stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")


## 1 · Data

Same instrument and windows as the original dashboard: **AA**, 5 days of 1-minute bars for the
intraday series and 1 year of daily bars to calibrate the bar sizes.  The loader tries `yfinance`
first (works out-of-the-box on Colab); if that is rate-limited or blocked it falls back to Yahoo's
public chart endpoint via `requests`, so the notebook always runs.

In [ ]:
TICKER   = "AA"      # ITA = US aerospace/defence ETF, DX-Y.NYB = dollar index, etc.
INTRADAY = ("5d", "1m")
DAILY    = ("1y", "1d")

def _flatten(df):
    if isinstance(df.columns, pd.MultiIndex):
        # yfinance single-ticker frames carry a redundant Ticker level
        lvl = "Ticker" if "Ticker" in df.columns.names else df.columns.names[-1]
        try:
            df = df.droplevel(lvl, axis=1)
        except Exception:
            df.columns = df.columns.get_level_values(0)
    return df[["Open", "High", "Low", "Close", "Volume"]].dropna()

def _via_yfinance(ticker, period, interval):
    import yfinance as yf
    df = yf.download(ticker, period=period, interval=interval,
                     progress=False, auto_adjust=True)
    return _flatten(df) if df is not None and len(df) else None

def _via_requests(ticker, rng, itv):
    import requests, time
    hdr = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"}
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{ticker}?range={rng}&interval={itv}"
    for _ in range(5):
        r = requests.get(url, headers=hdr, timeout=30)
        if r.status_code == 200:
            res = r.json()["chart"]["result"][0]
            q = res["indicators"]["quote"][0]
            df = pd.DataFrame(
                {"Open": q["open"], "High": q["high"], "Low": q["low"],
                 "Close": q["close"], "Volume": q["volume"]},
                index=pd.to_datetime(res["timestamp"], unit="s"))
            return df.dropna()
        time.sleep(6)
    raise RuntimeError(f"Yahoo chart fetch failed for {ticker}")

def load_ohlcv(ticker, period, interval, rng_itv):
    try:
        out = _via_yfinance(ticker, period, interval)
        if out is not None and len(out):
            return out
    except Exception as e:
        print(f"yfinance unavailable ({e}); using requests fallback.")
    return _via_requests(ticker, *rng_itv)

df  = load_ohlcv(TICKER, *INTRADAY, INTRADAY)   # 1-minute intraday
df2 = load_ohlcv(TICKER, *DAILY,    DAILY)      # 1-day, one year

# de Prado's rule of thumb: aim for ~50 bars per day -> daily reference buckets
bucket        = df2["Volume"].mean() / 50                       # volume bucket
dollar_bucket = (df2["Close"] * df2["Volume"]).mean() / 50      # dollar (equity) bucket
print(f"intraday rows: {len(df)}   daily rows: {len(df2)}")
print(f"volume bucket ~ {bucket:,.0f} sh   dollar bucket ~ {dollar_bucket:,.0f} $")
df.tail(3)


## 2 · Shared analytics (identical maths for every bar)

Everything the dashboard needs is packaged into a handful of functions so that the **regular-time
baseline** and **all three information bars** run through the *exact same* pipeline:

1. `ewma_garch_vol` — the original GARCH(1,1)/EWMA volatility (`λ = 0.2`).
2. `preprocess` — returns, price differences, GARCH volatility, z-score and the Student-t CDF that
   drives Bulk-Volume-Classification (`dof = 0.25`).
3. `compute_vpin` — BVC buy/sell split, **VPIN** and **pure VPIN** as a rolling mean over `window`
   buckets, the log-normal **confidence**, the **press** indicator, and the **moving averages**
   (fast/slow price MA + VPIN MA) — every window is measured in *buckets of the active clock*.

In [ ]:
LMBDA = 0.2      # EWMA / GARCH(1,1) decay used throughout
DOF   = 0.25     # Student-t d.o.f. for the BVC volume classifier

def ewma_garch_vol(diff, lmbda=LMBDA):
    """GARCH(1,1) special case (EWMA of squared price changes) -> volatility path."""
    diff = np.asarray(diff, dtype=float)
    var = np.empty(len(diff))
    var[0] = abs(diff[0])                                   # original initialisation
    for i in range(1, len(diff)):
        var[i] = lmbda * diff[i] ** 2 + (1 - lmbda) * var[i - 1]
    return np.sqrt(var)

def preprocess(data):
    """Returns, GARCH volatility, z-score and the Student-t CDF (BVC probability)."""
    data = data.copy()
    data["time"] = data.index                              # keep the real timestamp
    data["ret"]  = data["Close"].pct_change()
    data["diff"] = data["Close"].diff()
    data = data.dropna(subset=["diff"]).reset_index(drop=True)
    data["ewm_std"] = ewma_garch_vol(data["diff"].values)
    data["z_score"] = data["diff"] / data["ewm_std"]
    data["z_perc"]  = stats.t.cdf(data["z_score"], df=DOF)
    return data

def compute_vpin(data, window=60, ma_price_fast=10, ma_price_slow=30, ma_vpin=30, reset=True):
    """BVC -> VPIN / pure VPIN / confidence / press (+ moving averages), all in *buckets*.

    VPIN is written division-free as the rolling mean of the signed order-flow fraction
    (2*z_perc - 1).  This is algebraically identical to |V_buy - V_sell| / V whenever V > 0,
    but stays finite when a bar carries zero traded volume (illiquid tick bars), avoiding a
    0/0 -> NaN hole in the series.  `reset=False` preserves the post-warm-up integer index
    (used for the baseline so its x-axis matches the original exactly).
    """
    data = data.copy()
    zp = data["z_perc"]
    data["rbvolume"] = data["Volume"] * zp                 # bulk-classified BUY volume
    data["rsvolume"] = data["Volume"] - data["rbvolume"]   # bulk-classified SELL volume

    frac = 2.0 * zp - 1.0                                   # (V_buy - V_sell) / V, per bar
    data["vpin"]     = frac.abs().rolling(window).mean()
    data["purevpin"] = frac.rolling(window).mean()
    data = data.dropna(subset=["vpin"])
    if reset:
        data = data.reset_index(drop=True)

    data["log_vpin"] = np.log(data["vpin"].where(data["vpin"] > 0))   # NaN (gap) if ever 0
    data["vpin_confidence"] = stats.t.cdf(
        data["log_vpin"], df=window,
        loc=data["log_vpin"].mean(), scale=data["log_vpin"].std())

    data["direct"] = np.where(data["diff"] >= 0, 1, -1)
    data["press"]  = data["direct"].rolling(window).mean() # >0 => dominant buy pressure

    # --- moving averages with respect to the active bar clock ---
    data["price_ma_fast"] = data["Close"].rolling(ma_price_fast).mean()
    data["price_ma_slow"] = data["Close"].rolling(ma_price_slow).mean()
    data["purevpin_ma"]   = data["purevpin"].rolling(ma_vpin).mean()
    data["vpin_ma"]       = data["vpin"].rolling(ma_vpin).mean()
    data.attrs["ma"] = dict(price_fast=ma_price_fast, price_slow=ma_price_slow, vpin=ma_vpin)
    return data


## 3 · The dashboard (one function, reused for every bar)

`render_dashboard` reproduces the original figure **cell-for-cell**: the horizontal buy/sell
**volume profile on the left**, **price (cyan) on the left axis** with **confidence (colour-graded by
significance) on the right axis**, the 90/95/99 % confidence guide-lines, the purple **press** panel
and the green **pure-VPIN** panel with its dashed MA.  A single flag `show_price_ma` overlays the new
information-bar price moving averages.

In [ ]:
confidence_categories = {
    "Low":       {"min": 0.00, "max": 0.90, "color": "rgba(0, 0, 0, 0)"},  # transparent
    "Moderate":  {"min": 0.90, "max": 0.95, "color": "yellow"},
    "High":      {"min": 0.95, "max": 0.99, "color": "orange"},
    "Very High": {"min": 0.99, "max": 1.00, "color": "red"},
}

def get_confidence_category(v):
    if v < 0.90:   return "Low"
    elif v < 0.95: return "Moderate"
    elif v < 0.99: return "High"
    else:          return "Very High"

def volume_profile(test, lookback=60):
    """Buy/sell volume aggregated by $1 price level over the last `lookback` bars."""
    test = test.copy()
    test["1dp"] = np.floor(test["Close"])
    vol = test.tail(lookback)
    g = vol.groupby("1dp")[["rbvolume", "rsvolume"]].sum()
    g["rsvolume"] = -g["rsvolume"]                                   # plot sells to the left
    g["total_volume"] = g["rbvolume"] + abs(g["rsvolume"])
    g["buy_volume_percentage"]  = g["rbvolume"] / g["total_volume"] * 100
    g["sell_volume_percentage"] = abs(g["rsvolume"]) / g["total_volume"] * 100
    return g

def render_dashboard(test, bar_label="Regular time (1-min)", lookback=60,
                     show_price_ma=False, main_title=None, right_subtitle=None,
                     mid_subtitle="Press / Order-flow pressure", vpin_ma_label=None):
    """Exact replica of the original VPIN dashboard, parametrised by information bar."""
    test = test.copy()
    test["confidence_category"] = test["vpin_confidence"].apply(get_confidence_category)
    test["category_change"] = (test["confidence_category"]
                               != test["confidence_category"].shift()).cumsum()
    grouped_data = volume_profile(test, lookback)
    price_range = [test["Close"].min(), test["Close"].max()]

    fig = make_subplots(
        rows=3, cols=2, shared_xaxes=True, vertical_spacing=0.03, horizontal_spacing=0.05,
        row_heights=[0.5, 0.25, 0.25], column_widths=[0.3, 0.7],
        specs=[[{"type": "bar"}, {"secondary_y": True}], [None, {}], [None, {}]])

    # ---- left: buy / sell volume profile ----
    fig.add_trace(go.Bar(
        x=grouped_data["rbvolume"], y=grouped_data.index, name="Buy Volume", orientation="h",
        marker=dict(color="green"), customdata=grouped_data["buy_volume_percentage"],
        hovertemplate="Price: $%{y}<br>Buy Volume: %{x}<br>Buy %: %{customdata:.2f}%<extra></extra>"),
        row=1, col=1)
    fig.add_trace(go.Bar(
        x=grouped_data["rsvolume"], y=grouped_data.index, name="Sell Volume", orientation="h",
        marker=dict(color="red"), customdata=grouped_data["sell_volume_percentage"],
        hovertemplate="Price: $%{y}<br>Sell Volume: %{x}<br>Sell %: %{customdata:.2f}%<extra></extra>"),
        row=1, col=1)
    max_volume = max(grouped_data["rbvolume"].max(), -grouped_data["rsvolume"].min())
    fig.update_xaxes(range=[-max_volume * 1.1, max_volume * 1.1], zeroline=True,
                     zerolinecolor="white", zerolinewidth=2, title_text="Volume",
                     color="white", gridcolor="gray", row=1, col=1)
    fig.update_yaxes(title_text="Price Level ($)", color="white", gridcolor="gray",
                     range=price_range, row=1, col=1)

    # ---- right top: price (left axis) ----
    fig.add_trace(go.Scatter(
        x=test.index, y=test["Close"], name="Price", mode="lines",
        line=dict(color="cyan", dash="solid"), customdata=test["time"],
        hovertemplate="Index: %{x}<br>Price: $%{y}<br>Time: %{customdata|%Y-%m-%d %H:%M:%S}<extra></extra>"),
        row=1, col=2, secondary_y=False)

    # ---- moving averages with respect to information bars ----
    if show_price_ma:
        ma = test.attrs.get("ma", {})
        fig.add_trace(go.Scatter(
            x=test.index, y=test["price_ma_fast"], mode="lines",
            name=f"Price MA fast ({ma.get('price_fast', '')} bars)",
            line=dict(color="deepskyblue", dash="dot"),
            hovertemplate="Index: %{x}<br>Price MA fast: $%{y:.2f}<extra></extra>"),
            row=1, col=2, secondary_y=False)
        fig.add_trace(go.Scatter(
            x=test.index, y=test["price_ma_slow"], mode="lines",
            name=f"Price MA slow ({ma.get('price_slow', '')} bars)",
            line=dict(color="white", dash="dot"),
            hovertemplate="Index: %{x}<br>Price MA slow: $%{y:.2f}<extra></extra>"),
            row=1, col=2, secondary_y=False)

    # ---- right top: confidence (right axis), colour-graded by significance ----
    legend_categories = set()
    for _, group_data in test.groupby("category_change"):
        category = group_data["confidence_category"].iloc[0]
        color = confidence_categories[category]["color"]
        show_legend = category not in legend_categories
        if show_legend:
            legend_categories.add(category)
        fig.add_trace(go.Scatter(
            x=group_data.index, y=group_data["vpin_confidence"],
            name=f"Confidence ({category})" if show_legend else "", mode="lines",
            line=dict(color=color, dash="dot"), customdata=group_data["time"],
            showlegend=show_legend,
            hovertemplate=("Index: %{x}<br>Confidence: %{y:.2f}<br>"
                           "Time: %{customdata|%Y-%m-%d %H:%M:%S}<br>Category: "
                           + category + "<extra></extra>")),
            row=1, col=2, secondary_y=True)

    for level in [0.90, 0.95, 0.99]:
        fig.add_shape(type="line", x0=test.index.min(), x1=test.index.max(),
                      y0=level, y1=level, line=dict(color="white", width=2, dash="dash"),
                      xref="x", yref="y2", row=1, col=2)
        fig.add_annotation(x=test.index.max(), y=level, xref="x", yref="y2",
                           text=f"{int(level * 100)}% Confidence", showarrow=False,
                           xanchor="left", yanchor="bottom", font=dict(color="white"),
                           bgcolor="rgba(0,0,0,0.5)", row=1, col=2)

    # ---- middle: press ----
    fig.add_trace(go.Scatter(
        x=test.index, y=test["press"], name="Press", mode="lines",
        line=dict(color="purple", dash="solid"),
        hovertemplate="Index: %{x}<br>Press: %{y}<extra></extra>"), row=2, col=2)

    # ---- bottom: pure VPIN + MA ----
    ma_vpin = test.attrs.get("ma", {}).get("vpin", 30)
    fig.add_trace(go.Scatter(
        x=test.index, y=test["purevpin"], name="Pure VPIN", mode="lines",
        line=dict(color="green", dash="solid"), customdata=test["time"],
        hovertemplate="Index: %{x}<br>Pure VPIN: %{y}<br>Time: %{customdata|%Y-%m-%d %H:%M:%S}<extra></extra>"),
        row=3, col=2)
    fig.add_trace(go.Scatter(
        x=test.index, y=test["purevpin_ma"],
        name=vpin_ma_label if vpin_ma_label is not None else f"Pure VPIN MA ({ma_vpin} bars)",
        mode="lines", line=dict(color="red", dash="dash"),
        hovertemplate="Index: %{x}<br>Pure VPIN MA: %{y:.2f}<extra></extra>"), row=3, col=2)

    # ---- axes ----
    fig.update_xaxes(title_text="Index of DataFrame", color="white", gridcolor="gray", row=3, col=2)
    fig.update_yaxes(title_text="Price", secondary_y=False, title_font=dict(color="cyan"),
                     tickfont=dict(color="cyan"), gridcolor="gray", range=price_range, row=1, col=2)
    fig.update_yaxes(title_text="Confidence", secondary_y=True, title_font=dict(color="magenta"),
                     tickfont=dict(color="magenta"), range=[0, 1], dtick=0.05,
                     gridcolor="gray", row=1, col=2)
    fig.update_yaxes(title_text="Press", title_font=dict(color="purple"),
                     tickfont=dict(color="purple"), gridcolor="gray", row=2, col=2)
    fig.update_yaxes(title_text="Pure VPIN", title_font=dict(color="green"),
                     tickfont=dict(color="green"), gridcolor="gray", row=3, col=2)

    if main_title is None:
        main_title = (f"{bar_label} — Volume & Price with VPIN, Moving Average, and Press")
    fig.update_layout(
        title=dict(text=main_title, font=dict(color="white")),
        legend=dict(title=dict(text="Metrics"), font=dict(color="white")),
        hovermode="x unified", width=1800, height=1200,
        plot_bgcolor="black", paper_bgcolor="black", font=dict(color="white"),
        bargap=0.1, barmode="overlay")
    right_txt = (right_subtitle if right_subtitle is not None
                 else f"{bar_label}: Price, Confidence, and Pure VPIN Over Index")
    fig.update_layout(annotations=[
        dict(text="Buy and Sell Volume with Percentage by Price Level",
             x=0.15, y=1.2, showarrow=False, font=dict(size=16, color="white"),
             xref="paper", yref="paper"),
        dict(text=right_txt,
             x=0.85, y=1.2, showarrow=False, font=dict(size=16, color="white"),
             xref="paper", yref="paper"),
        dict(text=mid_subtitle, x=0.85, y=0.35, showarrow=False,
             font=dict(size=16, color="white"), xref="paper", yref="paper")])
    return fig


## 4 · Baseline — the original 1-minute dashboard

Running the shared pipeline on the raw 1-minute series with the original settings (`window = 60`,
VPIN MA = 30, 60-bar volume profile) reproduces **your original graph exactly**.

In [ ]:
# reset=False keeps the original post-warm-up index so the x-axis matches the original exactly
base = compute_vpin(preprocess(df), window=60, ma_vpin=30, reset=False)
fig_base = render_dashboard(
    base, bar_label="Regular time (1-min)", lookback=60, show_price_ma=False,
    main_title="Combined Volume and Price Chart with Pure VPIN, Moving Average, and Additional Plot",
    right_subtitle="Price, Confidence, and Pure VPIN Over Index",
    mid_subtitle="Additional Plot", vpin_ma_label="Pure VPIN MA (30)")
fig_base.show()


## 5 · de Prado information bars with GARCH-adjusted size

`build_information_bars` walks the 1-minute tape and closes a new bar whenever a running accumulator
crosses a threshold:

* **volume** bar → accumulator = share volume
* **dollar / equity** bar → accumulator = price × volume
* **tick / price** bar → accumulator = number of ticks

**GARCH adjusts the size.** Volatility is a proxy for the speed of information arrival, so the live
threshold is scaled by the GARCH/EWMA volatility measured on the minute tape:

$$\text{threshold}_t = \frac{\text{base}}{\operatorname{clip}\!\left(\sigma_t/\bar\sigma,\;0.5,\;2\right)}$$

High volatility ⇒ smaller bars (sample faster); low volatility ⇒ larger bars.  `base` is chosen so we
obtain roughly `target_bars` bars (de Prado's *constant-information* sampling), and it is reported
alongside the daily `bucket` reference from §1.

In [ ]:
def build_information_bars(df, bar_type="volume", target_bars=400, garch_adjust=True,
                           lmbda=LMBDA, vf_lo=0.5, vf_hi=2.0):
    """Resample a 1-minute tape into de Prado information bars with GARCH-adjusted thresholds."""
    df = df.copy()
    close  = df["Close"].astype(float).values
    volume = df["Volume"].astype(float).values
    if bar_type == "volume":
        inc = volume
    elif bar_type in ("dollar", "equity"):
        inc = close * volume
    elif bar_type in ("tick", "price"):
        inc = np.ones(len(df))
    else:
        raise ValueError(f"unknown bar_type: {bar_type}")

    base_threshold = inc.sum() / target_bars                     # ~ target_bars bars
    if not np.isfinite(base_threshold) or base_threshold <= 0:
        raise ValueError(
            f"Cannot size '{bar_type}' bars: cumulative {bar_type} value is zero for this "
            f"instrument (e.g. a volume-less index such as DX-Y.NYB). Use tick/price bars, "
            f"which do not require traded volume, or pick a ticker that reports volume.")

    diff = np.zeros(len(df)); diff[1:] = np.diff(close)
    gvol = ewma_garch_vol(diff, lmbda)                           # GARCH vol on the minute tape
    mean_vol = gvol.mean() or 1.0

    idx = df.index
    bars, run, start, thr = [], 0.0, 0, base_threshold
    for i in range(len(df)):
        if i == start:                                          # size fixed at bar open
            if garch_adjust:
                thr = base_threshold / np.clip(gvol[i] / mean_vol, vf_lo, vf_hi)
            else:
                thr = base_threshold
        run += inc[i]
        if run >= thr or i == len(df) - 1:                      # close the bar
            seg = df.iloc[start:i + 1]
            bars.append({
                "Open":  seg["Open"].iloc[0],  "High": seg["High"].max(),
                "Low":   seg["Low"].min(),     "Close": seg["Close"].iloc[-1],
                "Volume": seg["Volume"].sum(),
                "dollar": float((seg["Close"] * seg["Volume"]).sum()),
                "n_ticks": len(seg),
                "time": idx[i], "start_time": idx[start], "threshold": thr,
            })
            start, run = i + 1, 0.0

    out = pd.DataFrame(bars)
    out.index = pd.Index(out["time"], name="Datetime")          # timestamp index for preprocess()
    return out

# quick look at how many bars each clock produces
for bt in ["volume", "dollar", "tick"]:
    b = build_information_bars(df, bt, target_bars=400, garch_adjust=True)
    print(f"{bt:7s}: {len(b):4d} bars   median ticks/bar = {b['n_ticks'].median():.0f}")


## 6 · The same dashboard, one per information bar

Set `SELECTED_BARS` to a single clock to see just that one, or leave all three to render the **same
graph one after another**.  For every bar we:

1. build the GARCH-sized information bars,
2. run the identical `preprocess → compute_vpin` pipeline **on the bar clock** (so the VPIN, its
   buckets and rolling window are all in *bars*, exactly as de Prado's VPIN is defined),
3. render the identical dashboard, now with the **information-bar moving averages** overlaid.

In [ ]:
# ---------- choose your bar(s) ----------
SELECTED_BARS = ["volume", "dollar", "tick"]   # e.g. ["volume"] for a single clock
TARGET_BARS   = 400        # de Prado constant-information target (~ bars over the window)
VPIN_WINDOW   = 50         # VPIN buckets, in *bars* (de Prado uses ~50)
GARCH_ADJUST  = True       # GARCH-adjusted bar size
MA_PRICE_FAST, MA_PRICE_SLOW, MA_VPIN = 10, 30, 30   # moving-average windows, in *bars*
PROFILE_LOOKBACK = 60      # volume-profile depth, in *bars*

BAR_LABELS = {"volume": "Volume bars", "dollar": "Equity (dollar) bars", "tick": "Price (tick) bars"}

bar_results = {}
for bar_type in SELECTED_BARS:
    bars = build_information_bars(df, bar_type, target_bars=TARGET_BARS, garch_adjust=GARCH_ADJUST)
    proc = preprocess(bars)
    vpin = compute_vpin(proc, window=VPIN_WINDOW,
                        ma_price_fast=MA_PRICE_FAST, ma_price_slow=MA_PRICE_SLOW, ma_vpin=MA_VPIN)
    bar_results[bar_type] = vpin
    label = BAR_LABELS[bar_type]
    print(f"{label}: {len(bars)} bars -> {len(vpin)} VPIN points "
          f"(mean VPIN {vpin['vpin'].mean():.3f})")
    fig = render_dashboard(vpin, bar_label=label, lookback=PROFILE_LOOKBACK, show_price_ma=True)
    fig.show()


## 7 · Diagnostics on the selected bar clock

The original log-normality / dependence checks, now on the information-bar VPIN: the log-VPIN vs
absolute-return relationship, the Kolmogorov–Smirnov log-normality test, and the lagged
cross/auto-correlations that motivate the rolling window.

In [ ]:
DIAG_BAR = SELECTED_BARS[0]
d = bar_results[DIAG_BAR]
label = BAR_LABELS[DIAG_BAR]

# --- log-normality of VPIN (K-S test) ---
ks_stat, ks_p = stats.kstest(d["log_vpin"], "norm",
                             args=(d["log_vpin"].mean(), d["log_vpin"].std(ddof=0)))
print(f"[{label}] K-S statistic = {ks_stat:.4f}   p-value = {ks_p:.3e}")

# --- log-VPIN vs |return| ---
sc = go.Figure(go.Scatter(x=d["log_vpin"], y=d["ret"].abs(), mode="markers",
                          marker=dict(color="orange", size=5, opacity=0.6)))
sc.update_layout(template="plotly_dark", title=f"{label}: Log-VPIN vs |return|",
                 xaxis_title="log(VPIN)", yaxis_title="|return|", width=800, height=500)
sc.show()

# --- lagged correlation: log-VPIN vs FUTURE |return| (predictive direction), and VPIN autocorrelation ---
# NOTE: this uses shift(-lag) so it measures whether current VPIN LEADS future volatility
# (VPIN as a toxicity/early-warning signal). The original notebook used shift(+lag) (past
# returns); the forward direction here is the deliberate, more informative framing.
max_lag = min(120, len(d) // 3)
xcorr = [d["log_vpin"].corr(d["ret"].abs().shift(-lag)) for lag in range(max_lag + 1)]
acorr = [d["log_vpin"].corr(d["log_vpin"].shift(lag))   for lag in range(1, max_lag + 1)]
lc = make_subplots(rows=1, cols=2, subplot_titles=("log-VPIN vs future |return|",
                                                   "log-VPIN autocorrelation"))
lc.add_trace(go.Bar(x=list(range(max_lag + 1)), y=xcorr, marker_color="red"), row=1, col=1)
lc.add_trace(go.Bar(x=list(range(1, max_lag + 1)), y=acorr, marker_color="blue"), row=1, col=2)
lc.update_layout(template="plotly_dark", showlegend=False,
                 title=f"{label}: lagged correlation structure (lags in bars)",
                 width=1200, height=450)
lc.show()


## 8 · Method notes

* **BVC (Bulk Volume Classification).** Each bar's volume is split into buy/sell using the Student-t
  CDF of the GARCH-standardised price change, `z_perc = t.cdf(ΔP/σ_GARCH, dof=0.25)` — de Prado &
  Easley's symmetric classifier. `rbvolume = V·z_perc`, `rsvolume = V·(1 − z_perc)`.
* **VPIN.** `VPIN = ⟨|V_buy − V_sell| / V⟩` over `window` **bars**; *pure VPIN* keeps the sign.  On
  **volume bars** these are equal-volume buckets, i.e. the canonical volume-synchronised PIN; the
  dollar and tick clocks are the analogous constructions.  It is implemented division-free as
  `⟨|2·z_perc − 1|⟩` (identical to `|V_buy − V_sell| / V` for `V > 0`) so it stays finite even for a
  zero-volume bar; instruments with no volume at all raise a clear error at bar construction.
* **Confidence.** `t.cdf(log VPIN, df=window, loc=mean, scale=std)` — a live percentile of VPIN under
  a log-normal fit; the 90/95/99 % lines flag unusually toxic flow.
* **GARCH-adjusted size.** The bar threshold contracts when GARCH volatility rises, so sampling
  tracks the pace of information rather than the clock.
* **Moving averages.** Fast/slow price MAs and the VPIN MA all use windows counted **in bars**, so
  they are moving averages *with respect to the information bars*.
